# Phase 4 prep — pass-rate estimation over the free-form train pool

GRPO only learns from problems with **reward variance** in a generation group (pass rate strictly between
0 and all). This notebook measures each free-form train-pool problem's pass rate with the base model, so
the GRPO run trains on the **1/4–3/4 band** instead of all-correct (no gradient) or all-wrong (no gradient).

- Base `Qwen3-4B-Thinking-2507`, **no adapter**. Plain vLLM (same stack as eval).
- n=4 samples/problem, temperature 1.0 (matches GRPO rollout temp), cap 16384 (the intended train cap).
- Train pool only: excludes val / corrupted / unwinnable. **Val stays pristine.**
- **Resumable + checkpointed**: results append to Drive after every chunk; a disconnect loses ≤1 chunk.
- Writes `grpo_band.jsonl` (the 1/4–3/4 problems) for the GRPO notebook to train on, plus a raw
  per-problem record you can reuse without re-estimating.

Pure inference, fits on a 40GB A100. (The GRPO *training* run that consumes this wants an 80GB card to
train at 16k with a real group size.)

## 0. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/second_try/sft'          # data + harness/judger live here
GRPO_DIR    = '/content/drive/MyDrive/second_try/grpo'
import os, sys
os.makedirs(GRPO_DIR, exist_ok=True)
sys.path.insert(0, PROJECT_DIR)
print('PROJECT_DIR:', PROJECT_DIR)
print('GRPO_DIR   :', GRPO_DIR)

Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/second_try/sft
GRPO_DIR   : /content/drive/MyDrive/second_try/grpo


## 1. Install vLLM + grader deps (same stack as eval)

After this, **Runtime → Restart**, then run from section 2.

In [ ]:
!pip install -q uv 2>&1 | tail -1
!uv pip install --system torch==2.7.0 --index-url https://download.pytorch.org/whl/cu126
!uv pip install --system "vllm==0.9.2" "transformers==4.53.3" \
    sympy "antlr4-python3-runtime==4.11.1"
print("Install done. RESTART THE RUNTIME, then run from section 2.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 81.6 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 25 packages in 4.75s
Prepared 16 packages in 30.62s
Uninstalled 16 packages in 591ms
Installed 16 packages in 178ms
 - nvidia-cublas-cu12==12.8.4.1
 + nvidia-cublas-cu12==12.6.4.1
 - nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-cupti-cu12==12.6.80
 - nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-nvrtc-cu12==12.6.77
 - nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cuda-runtime-cu12==12.6.77
 - nvidia-cudnn-cu12==9.19.0.56
 + nvidia-cudnn-cu12==9.5.1.17
 - nvidia-cufft-cu12==11.3.3.83
 + nvidia-cufft-cu12==11.3.0.4
 - nvidia-cufile-cu12==1.13.1.3
 + nvidia-cufile-cu12==1.11.1.6
 - nvidia-curand-cu12==10.3.9.90
 + nvidia-curand-cu12==10.3.7.77
 - nvidia-cusolver-cu12==11.7.3.90
 + nvidia-cusolver-cu12==11.7.1.2
 - nvidia-cusparse-cu12==12.5.8.93
 + nvidia-cusparse-cu12==12.5.4.2
 - nvidia-cusparselt-cu12==0.7.1
 + nvidia-cusparselt-cu12==0.6.3
 - nvidia-nccl-c

## 2. Post-restart: version sanity + grading-path check

In [ ]:
import os, sys
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'

PROJECT_DIR = '/content/drive/MyDrive/second_try/sft'
GRPO_DIR    = '/content/drive/MyDrive/second_try/grpo'
sys.path.insert(0, PROJECT_DIR)

import torch, transformers, vllm
print('torch       :', torch.__version__)
print('transformers:', transformers.__version__)
print('vllm        :', vllm.__version__)
print('device      :', torch.cuda.get_device_name(0))
print('GPU free    :', round(torch.cuda.mem_get_info(0)[0] / 1e9, 2), 'GB')
assert transformers.__version__ == '4.53.3', 'wrong transformers — restart runtime'

from judger import Judger
_jt = Judger(strict_extract=False)
assert _jt.auto_judge(pred=r'\boxed{\frac{5}{8}}', gold=['5/8'], options=[[]]) is True, \
    'grading path broken — check sympy + antlr4-python3-runtime==4.11.1'
print('grading path: OK')

torch       : 2.7.0+cu126
transformers: 4.53.3
vllm        : 0.9.2
device      : NVIDIA A100-SXM4-40GB
GPU free    : 41.96 GB
grading path: OK


## 3. Config

In [ ]:
MODEL_ID     = 'Qwen/Qwen3-4B-Thinking-2507'    # base, no adapter
PUBLIC_PATH    = f'{PROJECT_DIR}/public.jsonl'
VAL_IDS_PATH   = f'{PROJECT_DIR}/val_ids.json'
CORRUPTED_PATH = f'{PROJECT_DIR}/corrupted_ids.json'
UNWINNABLE_PATH= f'{PROJECT_DIR}/unwinnable.json'

RAW_PATH  = f'{GRPO_DIR}/passrate_raw.jsonl'     # per-problem records (append-only, resumable)
BAND_PATH = f'{GRPO_DIR}/grpo_band.jsonl'        # the 1..n-1 learnable band -> GRPO train set

N_SAMPLES      = 4
TEMPERATURE    = 1.0
MAX_GEN_TOKENS = 16384
MAX_MODEL_LEN  = 20480       # prompt headroom + 16384 gen
GPU_MEM_UTIL   = 0.85
SEED           = 151
CHUNK          = 20          # problems per generate() call; checkpoint after each

print('config OK')
print(f'samples/problem={N_SAMPLES} temp={TEMPERATURE} cap={MAX_GEN_TOKENS}')
print(f'raw  -> {RAW_PATH}')
print(f'band -> {BAND_PATH}')

config OK
samples/problem=4 temp=1.0 cap=16384
raw  -> /content/drive/MyDrive/second_try/grpo/passrate_raw.jsonl
band -> /content/drive/MyDrive/second_try/grpo/grpo_band.jsonl


## 4. Build the free-form train-pool prompt set

In [ ]:
import json
import harness as H

SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Give your final answer inside a single \\boxed{}. "
    "Use EXACT values: prefer fractions (\\frac{a}{b}) and symbolic forms "
    "(\\sqrt{}, \\pi, e) over decimals. If you must give a decimal, write at "
    "least 10 significant figures and do NOT round. "
    "If the problem has multiple sub-answers, put them all inside one \\boxed{}, "
    "comma-separated, in the order asked, e.g. \\boxed{41, 35, 16}. "
    "If a single sub-answer itself contains a comma (a point or tuple), wrap it "
    "in parentheses, e.g. \\boxed{(2, 3), 7}."
)

def build_chat_ff(question):
    return [{'role':'system','content':SYSTEM_PROMPT_MATH},
            {'role':'user','content':question}]

data = H.load_jsonl(PUBLIC_PATH)
val_ids    = set(json.load(open(VAL_IDS_PATH)))
corrupted  = {c['id'] for c in json.load(open(CORRUPTED_PATH))}
unwinnable = {u['id'] for u in json.load(open(UNWINNABLE_PATH))}

ff_rows = [r for r in data
           if not r.get('options')                      # free-form only
           and r['id'] not in val_ids
           and r['id'] not in corrupted
           and r['id'] not in unwinnable]
row_map = {r['id']: r for r in data}
print(f'free-form train-pool problems to estimate: {len(ff_rows)}')
print(f'  total generations planned: {len(ff_rows) * N_SAMPLES}')

free-form train-pool problems to estimate: 636
  total generations planned: 2544


## 5a. Load base model

In [ ]:
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
llm = LLM(model=MODEL_ID, dtype='bfloat16', trust_remote_code=True,
          max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=GPU_MEM_UTIL,
          seed=SEED, enforce_eager=True)
sp = SamplingParams(n=N_SAMPLES, temperature=TEMPERATURE,
                    top_p=0.95, top_k=20, min_p=0.0,
                    max_tokens=MAX_GEN_TOKENS, seed=SEED)
print('model loaded')

INFO 05-30 15:11:10 [__init__.py:244] Automatically detected platform cuda.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

INFO 05-30 15:11:37 [config.py:841] This model supports multiple tasks: {'classify', 'embed', 'generate', 'reward'}. Defaulting to 'generate'.
INFO 05-30 15:11:37 [config.py:1472] Using max model len 20480
INFO 05-30 15:11:37 [config.py:2285] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 05-30 15:11:37 [cuda.py:102] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model loaded


## 5b. Generate + score in chunks (resumable)

Re-running this cell after a disconnect resumes where it left off — completed ids are skipped, results
are appended to `RAW_PATH` and fsync'd after every chunk. Safe to interrupt and restart.

In [ ]:
import json, os
from judger import Judger
_judger = Judger(strict_extract=False)

# Resume: load already-completed problem ids.
done_ids = set()
if os.path.exists(RAW_PATH):
    for line in open(RAW_PATH):
        line = line.strip()
        if line:
            done_ids.add(json.loads(line)['id'])
print(f'already done: {len(done_ids)}')

todo = [r for r in ff_rows if r['id'] not in done_ids]
print(f'remaining: {len(todo)}')

with open(RAW_PATH, 'a') as fout:
    for ci in range(0, len(todo), CHUNK):
        chunk = todo[ci:ci + CHUNK]
        prompts = [tok.apply_chat_template(build_chat_ff(r['question']),
                                           tokenize=False, add_generation_prompt=True)
                   for r in chunk]
        outs = llm.generate(prompts, sp)
        for r, out in zip(chunk, outs):
            samples = out.outputs                       # N_SAMPLES completions
            per = []
            for s in samples:
                trunc = (s.finish_reason == 'length')
                try:
                    diag = H.score_one(r, s.text, _judger, timeout=2)
                    correct = bool(diag['correct'])
                except Exception:
                    correct = False
                per.append({'correct': correct, 'truncated': trunc,
                            'n_tok': len(s.token_ids)})
            rec = {'id': r['id'], 'bucket': H.bucket_of(r),
                   'pass_count': sum(p['correct'] for p in per),
                   'n_samples': len(samples),
                   'n_truncated': sum(p['truncated'] for p in per),
                   'samples': per}
            fout.write(json.dumps(rec) + '\n')
        fout.flush(); os.fsync(fout.fileno())          # durable checkpoint
        print(f'  chunk done: {min(ci+CHUNK, len(todo))}/{len(todo)} (+{len(done_ids)} prior)')

print('generation + scoring complete')

already done: 250
remaining: 386


Adding requests:   0%|          | 0/20 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/80 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  chunk done: 20/386 (+250 prior)


Adding requests:   0%|          | 0/20 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/80 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  chunk done: 380/386 (+250 prior)


Adding requests:   0%|          | 0/6 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/24 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  chunk done: 386/386 (+250 prior)
generation + scoring complete


## 6. Aggregate → distribution + write band file

In [ ]:
import json
from collections import Counter

recs = [json.loads(l) for l in open(RAW_PATH) if l.strip()]
print(f'total problems estimated: {len(recs)}')

dist = Counter(r['pass_count'] for r in recs)
print('\npass-count distribution:')
for k in range(N_SAMPLES + 1):
    n = dist.get(k, 0)
    tag = '  (no gradient)' if k in (0, N_SAMPLES) else '  <- learnable band'
    print(f'  {k}/{N_SAMPLES}: {n:4d}{tag}')

band = [r for r in recs if 1 <= r['pass_count'] <= N_SAMPLES - 1]
print(f'\nlearnable band (1..{N_SAMPLES-1}): {len(band)} problems')

tot_samp = len(recs) * N_SAMPLES
tot_trunc = sum(r['n_truncated'] for r in recs)
print(f'truncated samples (hit {MAX_GEN_TOKENS} cap): {tot_trunc} / {tot_samp} '
      f'({100*tot_trunc/tot_samp:.1f}%)')
# How many problems are all-truncated (the cap is suppressing their pass rate)?
all_trunc = sum(1 for r in recs if r['n_truncated'] == r['n_samples'])
print(f'problems with ALL samples truncated: {all_trunc} '
      f'(these read as 0/{N_SAMPLES}; raise cap to recover them)')

# Write band with problem content for the GRPO trainer.
band_out = []
for r in band:
    pr = row_map[r['id']]
    band_out.append({'id': r['id'], 'question': pr['question'],
                     'answer': pr['answer'], 'pass_count': r['pass_count']})
with open(BAND_PATH, 'w') as f:
    for b in band_out:
        f.write(json.dumps(b) + '\n')
print(f'\nwrote {len(band_out)} band problems -> {BAND_PATH}')
band_buckets = Counter('free_multi' if len(b['answer']) > 1 else 'free_single' for b in band_out)
print('band bucket split:', dict(band_buckets))

total problems estimated: 636

pass-count distribution:
  0/4:  234  (no gradient)
  1/4:   14  <- learnable band
  2/4:   19  <- learnable band
  3/4:   25  <- learnable band
  4/4:  344  (no gradient)

learnable band (1..3): 58 problems
truncated samples (hit 16384 cap): 173 / 2544 (6.8%)
problems with ALL samples truncated: 21 (these read as 0/4; raise cap to recover them)

wrote 58 band problems -> /content/drive/MyDrive/second_try/grpo/grpo_band.jsonl
band bucket split: {'free_multi': 43, 'free_single': 15}


## 7. Next step

Report the pass-count distribution and the band size. Then I'll wire the GRPO notebook's §5 to load
`grpo_band.jsonl` directly (replacing the baseline-wrong filter) and tune `MAX_STEPS` to the band size.

Reading the distribution:
- **Healthy band (say 100+ in 1..3)** → GRPO has real signal to train on; proceed to the 80GB run at 16k.
- **Band dominated by 1/4, little 2–3/4** → problems are hard; GRPO can still work but expect slow reward gains.
- **Large `all samples truncated` count** → the 16384 cap is suppressing many problems to a false 0/4;
  those are long-reasoning problems we'd need a higher cap (and more VRAM) to train on.